In [1]:
from ultralytics import YOLO
from pathlib import Path
import os

In [2]:
BASE_DIR = Path("/mnt/f/CocDatasets/Perception1")
YOLO_DIR = BASE_DIR / "yolo_dataset_1"
DATA_YAML = YOLO_DIR / "data.yaml"

print(DATA_YAML.exists(), DATA_YAML)

True /mnt/f/CocDatasets/Perception1/yolo_dataset_1/data.yaml


In [3]:
train_imgs = list((YOLO_DIR / "images" / "train").glob("*"))
val_imgs = list((YOLO_DIR / "images" / "val").glob("*"))

train_lbls = list((YOLO_DIR / "labels" / "train").glob("*.txt"))
val_lbls = list((YOLO_DIR / "labels" / "val").glob("*.txt"))

print("Train images:", len(train_imgs))
print("Train labels:", len(train_lbls))
print("Val images:", len(val_imgs))
print("Val labels:", len(val_lbls))

Train images: 80
Train labels: 80
Val images: 20
Val labels: 20


In [4]:
sample_label = train_lbls[0]

with open(sample_label) as f:
    lines = f.readlines()

print("Sample label file:", sample_label.name)
print(lines[:3])

Sample label file: h10.txt
['0 0.387604 0.998750 0.407547 0.963296 0.434198 1.000000 0.412307 1.000000\n', '11 0.000000 0.257528 0.016271 0.273361 0.026286 0.323824 0.000000 0.350111\n', '3 0.001922 0.592333 0.011583 0.605213 0.023172 0.618944 0.025589 0.633546 0.012547 0.638694 0.000000 0.646417 0.000000 0.646130\n']


In [5]:
model = YOLO("yolov8s-seg.pt")

In [6]:
results = model.train(
    data=str(DATA_YAML),
    epochs=120,
    imgsz=1024,
    batch=8,
    workers=4,
    patience=25,
    device=0,
    project=str(BASE_DIR / "runs"),
    name="yolov8s_seg_perception_v1",
    pretrained=True,
    optimizer="auto",
    lr0=0.001,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1
)

New https://pypi.org/project/ultralytics/8.4.46 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.45 🚀 Python-3.10.20 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/f/CocDatasets/Perception1/yolo_dataset_1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s-se

In [ ]:
#resume training, tweak args
import torch

torch.cuda.empty_cache()

RUN_DIR = BASE_DIR / "runs" / "yolov8s_seg_perception_v1"

model = YOLO(RUN_DIR / "weights" / "last.pt")

results = model.train(
    resume=True,
    batch=4,      
    workers=2     
)

In [ ]:
best_model = YOLO(BASE_DIR / "runs" / "yolov8s_seg_perception" / "weights" / "best.pt")

metrics = best_model.val()
print(metrics)

In [ ]:
sample_imgs = [str(p) for p in val_imgs[:5]]

preds = best_model.predict(sample_imgs, save=True, conf=0.25)

In [ ]:
pred_dir = BASE_DIR / "runs" / "segment" / "predict"
print(pred_dir)
list(pred_dir.glob("*"))[:10]

In [ ]:
best_model.export(format="onnx")